# GLIDE vs FLEXPART footprint comparison

Side-by-side comparison of a GLIDE surface footprint against the FLEXPART reference fixture in `data/FLEXPART/`. Both are reduced to STILT-style surface sensitivity (`(mol/mol)/(mol/m²/s)`) on the **same lat/lon grid** (the GLIDE run is configured cell-for-cell against the fixture), so the panels are directly comparable.

GLIDE stores a raw residence-time footprint (`s` per cell); `lpdm.comparison.to_stilt_surface_footprint` converts it to the same units FLEXPART reports in `srr`. The surface layer matches on both sides (FLEXPART `LPDM_sampling_height = 40 m`, GLIDE bottom z-bin `0–40 m`), so the conversion is exact — no partial-overlap approximation.

This notebook is the comparison companion to `footprint_explorer.ipynb`; start here once a GLIDE run's `release_time`s overlap the fixture's.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr

import holoviews as hv
import geoviews as gv
import hvplot.xarray  # noqa: F401  registers the .hvplot accessor on DataArray

from lpdm.comparison import to_stilt_surface_footprint

hv.extension("bokeh")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GLIDE_ZARR = PROJECT_ROOT / "outputs" / "mhd-202401-hourly-gpu" / "footprints.zarr"
FLEXPART_NC = PROJECT_ROOT / "data" / "FLEXPART" / "FLEXPART_MHD_test_202401.nc"

# Surface-layer depth for the STILT conversion. Must match FLEXPART's
# LPDM_sampling_height and the GLIDE run's bottom z-bin edge (both 40 m here)
# for an exact (non-approximated) conversion.
SURFACE_LAYER_DEPTH_M = 40.0

print(f"GLIDE store:   {GLIDE_ZARR}  (exists: {GLIDE_ZARR.exists()})")
print(f"FLEXPART file: {FLEXPART_NC}  (exists: {FLEXPART_NC.exists()})")

## Load both datasets and find overlapping release times

The two products are matched on `release_time`. The FLEXPART fixture carries a subset of January 2024 (days 1, 10, 20, 30 × 24 h); only the GLIDE releases whose timestamps appear in both can be compared.

In [ ]:
glide_ds = xr.open_zarr(GLIDE_ZARR).load()
glide_fp = glide_ds["footprint"]

flex_ds = xr.open_dataset(FLEXPART_NC, engine="h5netcdf")
flex_srr = flex_ds["srr"]  # (time, latitude, longitude), units (mol/mol)/(mol/m^2/s)

release_lon = float(glide_ds.attrs["release_lon"])
release_lat = float(glide_ds.attrs["release_lat"])

common_times = np.intersect1d(glide_fp["release_time"].values, flex_srr["time"].values)
if common_times.size == 0:
    raise ValueError(
        "No overlapping release times between the GLIDE run and the FLEXPART fixture. "
        "Check that the GLIDE run covers one of the fixture days (1, 10, 20, 30 Jan 2024)."
    )

# Grids must match cell-for-cell for a direct comparison.
assert np.allclose(glide_fp["latitude"].values, flex_srr["latitude"].values), "latitude grids differ"
assert np.allclose(glide_fp["longitude"].values, flex_srr["longitude"].values), "longitude grids differ"

print(f"GLIDE releases:    {glide_fp.sizes['release_time']}")
print(f"FLEXPART releases: {flex_srr.sizes['time']}")
print(f"overlapping:       {common_times.size}")
print(f"first overlapping: {np.datetime_as_string(common_times[0], unit='h')}")
print(f"last overlapping:  {np.datetime_as_string(common_times[-1], unit='h')}")
print(f"release point:     ({release_lon:.4f}, {release_lat:.4f})")

## Select a release and convert GLIDE to STILT units

Change `RELEASE_INDEX` to compare a different overlapping release. The GLIDE→STILT conversion uses the near-surface air density FLEXPART reported for that release (derived from its `air_pressure`/`air_temperature`), so the two magnitudes are on a faithful common footing rather than a standard-atmosphere approximation.

In [ ]:
RELEASE_INDEX = 0  # index into common_times
release_time = common_times[RELEASE_INDEX]

# Near-surface air density from the FLEXPART-reported p, T at this release
# (air_pressure is hPa, air_temperature is K). rho = p / (R_d T).
R_DRY = 287.05
flex_at_release = flex_ds.sel(time=release_time)
p_pa = float(flex_at_release["air_pressure"]) * 100.0
T_k = float(flex_at_release["air_temperature"])
air_density = p_pa / (R_DRY * T_k)

glide_raw = glide_fp.sel(release_time=release_time)  # (time_ago, z_bin, lat, lon)
glide_stilt = to_stilt_surface_footprint(
    glide_raw,
    surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
    air_density_kg_m3=air_density,
    integrate_time=True,
)  # (lat, lon), units m^2 s mol^-1  ==  (mol/mol)/(mol/m^2/s)
flex_stilt = flex_srr.sel(time=release_time)  # (lat, lon), same units

print(f"release_time: {np.datetime_as_string(release_time, unit='h')}   air_density={air_density:.4f} kg/m^3")
print(f"GLIDE    total={float(glide_stilt.sum()):.4e}  max={float(glide_stilt.max()):.4e}  nonzero cells={int((glide_stilt > 0).sum())}")
print(f"FLEXPART total={float(flex_stilt.sum()):.4e}  max={float(flex_stilt.max()):.4e}  nonzero cells={int((flex_stilt > 0).sum())}")

## Side-by-side maps

Both panels share a log colour scale (computed from the union of the two fields' positive values) so colours mean the same sensitivity on each side. The cyan dot is the Mace Head release point. Pan/zoom are linked across the two maps.

In [ ]:
# Shared colour limits from the union of both positive fields.
glide_pos = glide_stilt.values[glide_stilt.values > 0]
flex_pos = flex_stilt.values[flex_stilt.values > 0]
both_pos = np.concatenate([glide_pos, flex_pos])
clim = (float(both_pos.min()), float(both_pos.max()))

release_marker = gv.Points([(release_lon, release_lat)]).opts(
    color="cyan", size=9, line_color="black",
)


def surface_map(field: xr.DataArray, title: str):
    """Log-scaled STILT surface-sensitivity raster on a tile basemap."""
    positive = field.where(field > 0)  # mask zeros so the log scale + transparency behave
    raster = positive.hvplot.image(
        x="longitude",
        y="latitude",
        tiles="CartoLight",
        cmap="magma",
        logz=True,
        clim=clim,
        geo=True,
        project=False,
        height=500,
        width=600,
        alpha=0.85,
        colorbar=True,
        clabel="surface sensitivity (mol/mol)/(mol/m\u00b2/s)",
        title=title,
    )
    return raster * release_marker


stamp = np.datetime_as_string(release_time, unit="h")
layout = (
    surface_map(glide_stilt, f"GLIDE  {stamp}")
    + surface_map(flex_stilt, f"FLEXPART  {stamp}")
).cols(2)
layout

## CH₄ enhancement timeseries — GLIDE vs all available LPDM runs

Modelled CH₄ enhancement (EDGAR 2024 annual-mean flux × footprint, summed over the domain → ppb) for one site, overlaying every available run:

- **GLIDE** — **all** release times in this run's footprint store (raw residence-time footprint → STILT units → × EDGAR). Not sub-sampled to the FLEXPART fixture.
- **FLEXPART / NAME** — the pre-computed validation timeseries in `data/validation-timeseries/` (full-year 2024 hourly): FLEXPART (ECMWF-HRES) and NAME under both met drivers (UMG, UKV).

$$\Delta c(t) = \sum_{i,j} f_{ij}(t)\,q_{ij}$$

with $f_{ij}$ in (mol/mol)/(mol m⁻² s⁻¹) and the EDGAR flux $q_{ij}$ in mol m⁻² s⁻¹. EDGAR 2024 is a single annual-mean field, so all temporal structure comes from the footprints.

Generate the validation CSVs first if the folder is empty: `python scripts/make_validation_timeseries.py`. Set `SITE_LABEL` to any inlet present there (e.g. `MHD-10magl`, `BSD-248magl`, `GAT-341magl`). The GLIDE line is overlaid when `SITE_LABEL` is the GLIDE release (Mace Head); it spans only the dates this GLIDE run covers.

In [ ]:
import pandas as pd  # noqa: E402

# Full GLIDE enhancement timeseries — EVERY release in the store (not sub-sampled to
# the FLEXPART-fixture overlap as the removed comparison was).
EDGAR_NC = PROJECT_ROOT / "data" / "EDGAR_CH4_2024_MHD_grid.nc"
edgar_flux = xr.open_dataset(EDGAR_NC)["flux"]  # (lat, lon), mol m-2 s-1

# Snap EDGAR onto the GLIDE footprint grid: they match to ~1e-14 (asserted in cmp-03)
# but xarray aligns on *exactly* equal labels, so a bare product would silently zero
# all but the bit-identical cells.
ref_lat = glide_fp["latitude"].values
ref_lon = glide_fp["longitude"].values
edgar_flux = edgar_flux.assign_coords(latitude=ref_lat, longitude=ref_lon)

# Representative near-surface air density for the raw→STILT conversion. The enhancement
# scales linearly with this; a per-release value from the run's met store would be exact
# (and the maps above use FLEXPART's reported p,T), but a single value is fine here and,
# unlike FLEXPART p,T, is available for every GLIDE release.
GLIDE_AIR_DENSITY_KG_M3 = 1.2

glide_times = glide_fp["release_time"].values
glide_ppb = []
for t in glide_times:
    stilt_t = to_stilt_surface_footprint(
        glide_fp.sel(release_time=t),
        surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
        air_density_kg_m3=GLIDE_AIR_DENSITY_KG_M3,
        integrate_time=True,
    )
    glide_ppb.append(float((stilt_t * edgar_flux).sum()) * 1e9)

glide_ts = pd.DataFrame(
    {"time": glide_times.astype("datetime64[ns]"), "ch4_enhancement_ppb": glide_ppb}
)
print(
    f"GLIDE: {len(glide_ts)} releases  "
    f"{np.datetime_as_string(glide_times[0], unit='h')} .. "
    f"{np.datetime_as_string(glide_times[-1], unit='h')}  "
    f"mean={glide_ts['ch4_enhancement_ppb'].mean():.2f}  "
    f"max={glide_ts['ch4_enhancement_ppb'].max():.1f} ppb"
)

In [ ]:
import pandas as pd
import hvplot.pandas  # noqa: F401  registers the .hvplot accessor on DataFrame
from functools import reduce

VTS_DIR = PROJECT_ROOT / "data" / "validation-timeseries"

# Site to overlay. Default = Mace Head (the GLIDE release above); change to any
# inlet label present in data/validation-timeseries/.
SITE_LABEL = "MHD-10magl"


def load_validation_runs(site_label: str) -> dict[str, pd.DataFrame]:
    """All available {model-driver: DataFrame} validation timeseries for one site.

    Filenames are ``{label}_{model}_{met}.csv``; we strip the (possibly
    underscore-containing) label prefix so e.g. ``MHD-10magl_NAME_UMG`` -> ``NAME-UMG``.
    Times are parsed to tz-naive UTC to share an axis with the in-notebook GLIDE series.
    """
    runs: dict[str, pd.DataFrame] = {}
    if not VTS_DIR.is_dir():
        return runs
    for path in sorted(VTS_DIR.glob(f"{site_label}_*.csv")):
        run_name = path.stem[len(site_label) + 1:].replace("_", "-")
        df = pd.read_csv(path, comment="#")
        df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
        runs[run_name] = df
    return runs


runs = load_validation_runs(SITE_LABEL)
if not runs:
    print(
        f"No validation timeseries for {SITE_LABEL!r} in {VTS_DIR}.\n"
        "Generate them first:  python scripts/make_validation_timeseries.py"
    )
else:
    print(f"{SITE_LABEL}: {len(runs)} run(s)")
    for name, df in runs.items():
        col = df["ch4_enhancement_ppb"]
        print(f"  {name:20s} {len(df):5d} h   mean={col.mean():6.2f}  max={col.max():7.1f} ppb")

In [ ]:
# Distinct colour per model/met-driver; unknown runs fall back to the hvplot cycle.
RUN_COLORS = {
    "FLEXPART-ECMWFHRES": "#3182bd",  # blue
    "NAME-UMG":           "#31a354",  # green
    "NAME-UKV":           "#756bb1",  # purple
}

lines = [
    df.hvplot.line(
        x="time", y="ch4_enhancement_ppb",
        label=name, color=RUN_COLORS.get(name), line_width=1.4,
    )
    for name, df in runs.items()
]

# Overlay the FULL GLIDE series (all release times) when the selected site is the GLIDE
# release (Mace Head). Drawn bolder; it spans only the dates this GLIDE run covers.
if SITE_LABEL.startswith("MHD") and "glide_ts" in globals() and len(glide_ts):
    lines.append(
        glide_ts.hvplot.line(
            x="time", y="ch4_enhancement_ppb",
            label="GLIDE", color="#e6550d", line_width=2.5,
        )
    )

if not lines:
    raise ValueError(
        f"No runs to plot for {SITE_LABEL!r}. Run scripts/make_validation_timeseries.py first."
    )

overlay = reduce(lambda a, b: a * b, lines)
overlay.opts(
    xlabel="Time (UTC)",
    ylabel="ΔCH₄ (ppb)",
    title=f"CH₄ enhancement at {SITE_LABEL} — EDGAR 2024 flux × footprint (all available LPDM runs)",
    legend_position="top_right",
    height=430,
    width=1000,
    toolbar="above",
)